# Практическая работа №3
## Прогнозирование энергопотребления с помощью LSTM и GRU в PyTorch

**Студентка:** Бахорина Ольга Александровна  
**Группа:** R4176c  
**Курс:** 1 курс магистратуры  
**Дисциплина:** «Прикладной искусственный интеллект»

В работе прогнозируется почасовое энергопотребление American Electric
Power на один час вперёд. Используется набор Hourly Energy Consumption
с Kaggle.

Выбран один домен — **временной ряд энергопотребления**. Сравниваются
LSTM и GRU; лучшая конфигурация выбирается только по validation, а
test используется после выбора модели для независимой итоговой оценки.


## 1. Цель и постановка задачи

**Цель:** освоить полный цикл прогнозирования временного ряда при помощи
рекуррентных нейронных сетей PyTorch.

**Задача:** по предыдущим 168 часам наблюдений предсказать нагрузку
энергосети AEP на следующий час, измеряемую в мегаваттах.

Этапы:

1. скачать датасет и проверить качество данных;
2. исследовать тренд и сезонность;
3. выполнить хронологическое разбиение 70/15/15;
4. нормализовать данные без утечки информации;
5. сформировать скользящие окна отдельно внутри каждого split;
6. подобрать гиперпараметры LSTM/GRU только по validation;
7. выполнить полное обучение на Kaggle GPU;
8. оценить train, validation и test по MAE, RMSE и R²;
9. сравнить RNN с простыми baseline;
10. сохранить checkpoint, scaler, конфигурацию и предсказания;
11. проанализировать неудачные прогнозы;
12. показать прогноз для следующего часа.

Текстовый домен является альтернативной веткой и в эту работу не входит.
Transformer относится к опциональному расширению и также не включён.


## 2. Подготовка окружения

In [ ]:
import copy
import hashlib
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
sns.set_theme(style="whitegrid")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


In [ ]:
DATASET_SLUG = "robikscube/hourly-energy-consumption"
SERIES_FILE = "AEP_hourly.csv"
TARGET = "AEP_MW"

LOOKBACK = 168
HORIZON = 1

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

BATCH_SIZE = 256
SEARCH_STRIDE = 3
SEARCH_EPOCHS = 6
MAX_EPOCHS = 25
NUM_WORKERS = 0

RUN_MODE = "full"
IS_KAGGLE = Path("/kaggle/working").is_dir()

if RUN_MODE == "full":
    if not IS_KAGGLE:
        raise RuntimeError(
            "Итоговый full-run должен выполняться в Kaggle."
        )

    if DEVICE.type != "cuda":
        raise RuntimeError(
            "Для итогового full-run требуется Kaggle GPU."
        )

artifact_root = (
    Path("/kaggle/working/lab3_artifacts")
    if IS_KAGGLE
    else Path("lab3_artifacts")
)
ARTIFACT_DIR = artifact_root
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_ID = (
    pd.Timestamp.now(tz="UTC")
    .strftime("%Y%m%dT%H%M%SZ")
    + "_p3_aep_rnn"
)

environment_info = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "device": str(DEVICE),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "seed": SEED,
}

if torch.cuda.is_available():
    environment_info["gpu"] = (
        torch.cuda.get_device_name(0)
    )
    environment_info["gpu_memory_bytes"] = (
        torch.cuda.get_device_properties(
            0
        ).total_memory
    )

(
    ARTIFACT_DIR
    / "environment.json"
).write_text(
    json.dumps(
        environment_info,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        environment_info,
        indent=2,
        ensure_ascii=False,
    )
)


## 3. Загрузка и описание датасета

Используется **Hourly Energy Consumption**, Kaggle Version 3.
Источник данных — PJM; в работе выбран ряд American Electric Power
(`AEP_hourly.csv`).

Датасет содержит более десяти лет оценок почасового энергопотребления
в мегаваттах.

Поля:

- `Datetime` — временная метка;
- `AEP_MW` — энергопотребление American Electric Power, МВт.

Лицензия набора на Kaggle: **CC0: Public Domain**.

Страница набора:
`https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption`


In [ ]:
downloaded_dir = Path(
    kagglehub.dataset_download(
        DATASET_SLUG
    )
)

candidates = list(
    downloaded_dir.rglob(
        SERIES_FILE
    )
)

if len(candidates) != 1:
    raise FileNotFoundError(
        f"Ожидался один файл {SERIES_FILE}, "
        f"найдено: {len(candidates)}"
    )

DATA_PATH = candidates[0]

raw_df = pd.read_csv(
    DATA_PATH
)
raw_df["Datetime"] = pd.to_datetime(
    raw_df["Datetime"],
    errors="raise",
)

source_info = {
    "dataset": "Hourly Energy Consumption",
    "kaggle_slug": DATASET_SLUG,
    "kaggle_version": 3,
    "license": "CC0: Public Domain",
    "file": SERIES_FILE,
    "sha256": hashlib.sha256(
        DATA_PATH.read_bytes()
    ).hexdigest(),
    "rows": int(len(raw_df)),
}

(
    ARTIFACT_DIR
    / "dataset_source.json"
).write_text(
    json.dumps(
        source_info,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

display(raw_df.head())
print(
    json.dumps(
        source_info,
        indent=2,
        ensure_ascii=False,
    )
)


In [ ]:
quality_before = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "unique": raw_df.nunique(),
})
display(quality_before)
display(raw_df[TARGET].describe().to_frame().T)

print("Период:", raw_df["Datetime"].min(), "—", raw_df["Datetime"].max())
print("Дубли временных меток:", raw_df["Datetime"].duplicated().sum())
print("Пропуски целевой переменной:", raw_df[TARGET].isna().sum())


## 4. Очистка и восстановление регулярной временной шкалы

Строки сортируются по времени. Повторяющиеся временные метки
агрегируются средним. Затем создаётся полный почасовой индекс.

Для восстановления редких пропусков используется только последнее
известное прошлое значение (`forward fill`). Будущие наблюдения
для заполнения пропусков не используются.


In [ ]:
series_df = (
    raw_df[["Datetime", TARGET]]
    .dropna(subset=["Datetime"])
    .groupby(
        "Datetime",
        as_index=True,
    )[TARGET]
    .mean()
    .sort_index()
    .to_frame()
)

full_index = pd.date_range(
    series_df.index.min(),
    series_df.index.max(),
    freq="1h",
)

missing_hours_before = len(
    full_index.difference(
        series_df.index
    )
)

series_df = series_df.reindex(
    full_index
)

missing_before_fill = int(
    series_df[TARGET]
    .isna()
    .sum()
)

series_df[TARGET] = (
    series_df[TARGET]
    .ffill()
)

series_df = series_df.dropna(
    subset=[TARGET]
)
series_df.index.name = "Datetime"

assert series_df.index.is_monotonic_increasing
assert (
    series_df.index.duplicated().sum()
    == 0
)
assert not series_df[TARGET].isna().any()

cleanup_info = {
    "missing_hours_before": (
        missing_hours_before
    ),
    "filled_from_past": (
        missing_before_fill
    ),
    "rows_after_cleanup": len(
        series_df
    ),
    "frequency": "1h",
}

(
    ARTIFACT_DIR
    / "cleanup.json"
).write_text(
    json.dumps(
        cleanup_info,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Размер очищенного ряда:",
    series_df.shape,
)
print(
    "Восстановлено отсутствующих часов:",
    missing_hours_before,
)
display(series_df.head())


## 5. Разведочный анализ

Для энергопотребления ожидаются сильные суточные и недельные циклы, различия между рабочими и выходными днями и сезонные изменения в течение года.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

series_df[TARGET].plot(ax=axes[0], linewidth=0.55, color="tab:blue")
axes[0].set_title("Почасовое энергопотребление AEP за весь период")
axes[0].set_ylabel("МВт")

series_df[TARGET].iloc[:24 * 30].plot(
    ax=axes[1], linewidth=1.0, color="tab:orange"
)
axes[1].set_title("Первый месяц: суточная и недельная динамика")
axes[1].set_ylabel("МВт")

sns.histplot(series_df[TARGET], bins=60, kde=True, ax=axes[2], color="tab:green")
axes[2].set_title("Распределение нагрузки")
axes[2].set_xlabel("Энергопотребление, МВт")

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "eda_energy.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
calendar_df = pd.DataFrame({
    "load": series_df[TARGET],
    "hour": series_df.index.hour,
    "day_of_week": series_df.index.dayofweek,
    "month": series_df.index.month,
})

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
calendar_df.groupby("hour")["load"].mean().plot(
    ax=axes[0], marker="o", color="tab:purple"
)
axes[0].set_title("Средняя нагрузка по часу")
axes[0].set_xlabel("Час")
axes[0].set_ylabel("МВт")

calendar_df.groupby("day_of_week")["load"].mean().plot(
    ax=axes[1], marker="o", color="tab:red"
)
axes[1].set_title("Средняя нагрузка по дню недели")
axes[1].set_xlabel("0 = понедельник")

calendar_df.groupby("month")["load"].mean().plot(
    ax=axes[2], marker="o", color="tab:green"
)
axes[2].set_title("Средняя нагрузка по месяцу")
axes[2].set_xlabel("Месяц")

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "seasonality.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
lags = np.arange(1, 24 * 14 + 1)
autocorrelation = [series_df[TARGET].autocorr(lag=int(lag)) for lag in lags]

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(lags, autocorrelation, linewidth=1.0)
for lag, label in [(24, "24 ч"), (168, "168 ч"), (336, "336 ч")]:
    ax.axvline(lag, color="tab:red", linestyle="--", alpha=0.6)
    ax.text(lag + 2, 0.05, label, rotation=90)
ax.set_title("Автокорреляция ряда для лагов до двух недель")
ax.set_xlabel("Лаг, часы")
ax.set_ylabel("Корреляция")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "autocorrelation.png", dpi=160, bbox_inches="tight")
plt.show()

**Промежуточный вывод.** Повторяющаяся форма почасового графика и пики автокорреляции около 24 и 168 часов подтверждают суточную и недельную сезонности. Поэтому окно модели выбрано равным одной неделе.

## 6. Календарные признаки и хронологическое разбиение

К исходной нагрузке добавляются циклические признаки часа, дня недели и дня года. Синус и косинус корректно отражают близость конца и начала каждого цикла. Они вычисляются только из временной метки и не содержат будущих значений ряда.

Первые 70% наблюдений используются для обучения, следующие 15% — для validation, последние 15% — для test. Случайное разбиение временного ряда не применяется.

In [ ]:
def add_calendar_features(df):
    result = df.copy()
    hour = result.index.hour.to_numpy()
    day_of_week = result.index.dayofweek.to_numpy()
    day_of_year = result.index.dayofyear.to_numpy()

    result["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    result["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    result["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    result["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7)
    result["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    result["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    return result

feature_df = add_calendar_features(series_df)
FEATURES = feature_df.columns.tolist()
display(feature_df.head())
print("Признаки:", FEATURES)


In [ ]:
n_total = len(feature_df)
train_end = int(
    n_total * TRAIN_RATIO
)
val_end = int(
    n_total
    * (
        TRAIN_RATIO
        + VAL_RATIO
    )
)

split_table = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "observations": [
            train_end,
            val_end - train_end,
            n_total - val_end,
        ],
        "start": [
            feature_df.index[0],
            feature_df.index[
                train_end
            ],
            feature_df.index[
                val_end
            ],
        ],
        "end": [
            feature_df.index[
                train_end - 1
            ],
            feature_df.index[
                val_end - 1
            ],
            feature_df.index[-1],
        ],
    }
)

split_table.to_csv(
    ARTIFACT_DIR
    / "split_manifest.csv",
    index=False,
)

display(split_table)

fig, ax = plt.subplots(
    figsize=(15, 5)
)

ax.plot(
    series_df.index,
    series_df[TARGET],
    linewidth=0.55,
)

ax.axvline(
    series_df.index[
        train_end
    ],
    linestyle="--",
    label="validation",
)

ax.axvline(
    series_df.index[
        val_end
    ],
    linestyle="--",
    label="test",
)

ax.set_title(
    "Хронологическое разбиение"
)
ax.set_ylabel("МВт")
ax.legend()

plt.tight_layout()
plt.savefig(
    ARTIFACT_DIR
    / "chronological_split.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 7. Нормализация без утечки и скользящие окна

`StandardScaler` обучается исключительно на train.

Скользящие окна строятся отдельно внутри каждого split:
train, validation и test не используют входные окна, пересекающие
границы соседних частей ряда.

Один вход имеет форму `(168, 7)`: недельная история нагрузки и
календарных признаков. Цель — нагрузка следующего часа.
Для каждого объекта выполняется проверка `input_end < target_time`.


In [ ]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

feature_scaler.fit(
    feature_df.iloc[
        :train_end
    ][FEATURES]
)
target_scaler.fit(
    feature_df.iloc[
        :train_end
    ][[TARGET]]
)

features_scaled = (
    feature_scaler.transform(
        feature_df[FEATURES]
    )
    .astype("float32")
)

target_scaled = (
    target_scaler.transform(
        feature_df[[TARGET]]
    )
    .astype("float32")
    .ravel()
)

joblib.dump(
    feature_scaler,
    ARTIFACT_DIR
    / "feature_scaler.joblib",
)
joblib.dump(
    target_scaler,
    ARTIFACT_DIR
    / "target_scaler.joblib",
)

print(
    "Средние train-признаков:",
    np.round(
        features_scaled[
            :train_end
        ].mean(axis=0),
        4,
    ),
)

print(
    "Стандартные отклонения "
    "train-признаков:",
    np.round(
        features_scaled[
            :train_end
        ].std(axis=0),
        4,
    ),
)


In [ ]:
class WindowDataset(Dataset):
    def __init__(
        self,
        features,
        target,
        target_start,
        target_end,
        lookback=LOOKBACK,
        horizon=HORIZON,
    ):
        self.features = features
        self.target = target
        self.target_start = target_start
        self.target_end = target_end
        self.lookback = lookback
        self.horizon = horizon

        first_target = (
            target_start
            + lookback
            + horizon
            - 1
        )

        self.target_indices = np.arange(
            first_target,
            target_end,
        )

    def __len__(self):
        return len(
            self.target_indices
        )

    def __getitem__(
        self,
        item,
    ):
        target_idx = int(
            self.target_indices[
                item
            ]
        )

        window_end = (
            target_idx
            - self.horizon
            + 1
        )
        window_start = (
            window_end
            - self.lookback
        )

        if (
            window_start
            < self.target_start
        ):
            raise RuntimeError(
                "Окно пересекает "
                "границу split."
            )

        if window_end > target_idx:
            raise RuntimeError(
                "Вход содержит "
                "будущее значение."
            )

        x_value = self.features[
            window_start:window_end
        ]
        y_value = self.target[
            target_idx
        ]

        return x_value, y_value


features_tensor = torch.from_numpy(
    features_scaled
)
target_tensor = torch.from_numpy(
    target_scaled
)

train_dataset = WindowDataset(
    features_tensor,
    target_tensor,
    0,
    train_end,
)
val_dataset = WindowDataset(
    features_tensor,
    target_tensor,
    train_end,
    val_end,
)
test_dataset = WindowDataset(
    features_tensor,
    target_tensor,
    val_end,
    n_total,
)

sample_x, sample_y = train_dataset[
    0
]

print(
    "Один объект:",
    sample_x.shape,
    "цель:",
    sample_y.shape,
)

print(
    "Количество окон:",
    {
        "train": len(
            train_dataset
        ),
        "validation": len(
            val_dataset
        ),
        "test": len(
            test_dataset
        ),
    },
)

assert sample_x.shape == (
    LOOKBACK,
    len(FEATURES),
)

assert (
    train_dataset.target_indices.max()
    < val_dataset.target_indices.min()
)

assert (
    val_dataset.target_indices.max()
    < test_dataset.target_indices.min()
)

assert (
    val_dataset.target_indices[0]
    - LOOKBACK
    >= train_end
)

assert (
    test_dataset.target_indices[0]
    - LOOKBACK
    >= val_end
)


In [ ]:
loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": (
        DEVICE.type == "cuda"
    ),
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=(
        torch.Generator()
        .manual_seed(SEED)
    ),
    **loader_kwargs,
)

train_eval_loader = DataLoader(
    train_dataset,
    shuffle=False,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)
test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs,
)

search_subset = Subset(
    train_dataset,
    range(
        0,
        len(train_dataset),
        SEARCH_STRIDE,
    ),
)

search_loader = DataLoader(
    search_subset,
    shuffle=True,
    generator=(
        torch.Generator()
        .manual_seed(SEED)
    ),
    **loader_kwargs,
)


## 8. Архитектура RNN и функции обучения

Проверяются LSTM и GRU. Последнее скрытое состояние проходит через LayerNorm, полносвязный блок и dropout. Используются Huber loss, AdamW, gradient clipping, ReduceLROnPlateau и EarlyStopping. Это стабилизирует обучение и уменьшает риск переобучения.

In [ ]:
class RecurrentRegressor(nn.Module):
    def __init__(
        self,
        input_size,
        cell="LSTM",
        hidden_size=64,
        num_layers=1,
        dropout=0.2,
    ):
        super().__init__()

        rnn_class = {
            "LSTM": nn.LSTM,
            "GRU": nn.GRU,
        }[cell]

        recurrent_dropout = (
            dropout
            if num_layers > 1
            else 0.0
        )

        self.rnn = rnn_class(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(
                hidden_size
            ),
            nn.Linear(
                hidden_size,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                32,
                1,
            ),
        )

    def forward(self, x_value):
        sequence, _ = self.rnn(
            x_value
        )

        return self.head(
            sequence[:, -1]
        ).squeeze(-1)


def make_model(config):
    return RecurrentRegressor(
        input_size=len(
            FEATURES
        ),
        cell=config["cell"],
        hidden_size=(
            config["hidden_size"]
        ),
        num_layers=(
            config["num_layers"]
        ),
        dropout=config["dropout"],
    ).to(DEVICE)


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
):
    training = (
        optimizer is not None
    )
    model.train(training)

    total_loss = 0.0
    total_abs_error = 0.0
    total_squared_error = 0.0
    total_count = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(
            DEVICE,
            non_blocking=True,
        )
        y_batch = y_batch.to(
            DEVICE,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):
            predictions = model(
                x_batch
            )
            loss = criterion(
                predictions,
                y_batch,
            )

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
                optimizer.step()

        errors = (
            predictions
            - y_batch
        )

        batch_size = y_batch.size(0)
        total_loss += (
            loss.item()
            * batch_size
        )
        total_abs_error += (
            errors.abs()
            .sum()
            .item()
        )
        total_squared_error += (
            errors.square()
            .sum()
            .item()
        )
        total_count += batch_size

    return {
        "loss": (
            total_loss
            / total_count
        ),
        "mae_scaled": (
            total_abs_error
            / total_count
        ),
        "rmse_scaled": np.sqrt(
            total_squared_error
            / total_count
        ),
    }


In [ ]:
def save_training_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    config,
    epoch,
    history,
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
        ),
        "config": config,
        "features": FEATURES,
        "lookback": LOOKBACK,
        "horizon": HORIZON,
        "run_id": RUN_ID,
        "seed": SEED,
        "history": history,
        "python_random_state": (
            random.getstate()
        ),
        "numpy_random_state": (
            np.random.get_state()
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
    }

    if torch.cuda.is_available():
        checkpoint[
            "cuda_rng_state_all"
        ] = (
            torch.cuda
            .get_rng_state_all()
        )

    torch.save(
        checkpoint,
        path,
    )


def train_model(
    config,
    train_dl,
    val_dl,
    epochs,
    patience,
    checkpoint_path=None,
    seed=SEED,
):
    set_seed(seed)

    model = make_model(
        config
    )
    criterion = nn.HuberLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ],
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        )
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_mae_scaled": [],
        "val_mae_scaled": [],
        "train_rmse_mw": [],
        "val_rmse_mw": [],
        "lr": [],
    }

    best_val_rmse = float(
        "inf"
    )
    best_state = None
    epochs_without_improvement = 0
    started = time.perf_counter()

    target_scale = float(
        target_scaler.scale_[0]
    )

    for epoch in range(
        1,
        epochs + 1,
    ):
        train_metrics = run_epoch(
            model,
            train_dl,
            criterion,
            optimizer,
        )

        with torch.no_grad():
            val_metrics = run_epoch(
                model,
                val_dl,
                criterion,
            )

        train_rmse_mw = (
            train_metrics[
                "rmse_scaled"
            ]
            * target_scale
        )
        val_rmse_mw = (
            val_metrics[
                "rmse_scaled"
            ]
            * target_scale
        )

        scheduler.step(
            val_rmse_mw
        )

        history[
            "train_loss"
        ].append(
            train_metrics["loss"]
        )
        history[
            "val_loss"
        ].append(
            val_metrics["loss"]
        )
        history[
            "train_mae_scaled"
        ].append(
            train_metrics[
                "mae_scaled"
            ]
        )
        history[
            "val_mae_scaled"
        ].append(
            val_metrics[
                "mae_scaled"
            ]
        )
        history[
            "train_rmse_mw"
        ].append(
            train_rmse_mw
        )
        history[
            "val_rmse_mw"
        ].append(
            val_rmse_mw
        )
        history["lr"].append(
            optimizer.param_groups[
                0
            ]["lr"]
        )

        improved = (
            val_rmse_mw
            < best_val_rmse
            - 1e-6
        )

        if improved:
            best_val_rmse = (
                val_rmse_mw
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0

            if checkpoint_path is not None:
                save_training_checkpoint(
                    checkpoint_path,
                    model,
                    optimizer,
                    scheduler,
                    config,
                    epoch,
                    history,
                )
        else:
            epochs_without_improvement += 1

        if checkpoint_path is not None:
            latest_path = (
                checkpoint_path
                .with_name(
                    "latest_checkpoint.pt"
                )
            )

            save_training_checkpoint(
                latest_path,
                model,
                optimizer,
                scheduler,
                config,
                epoch,
                history,
            )

        print(
            f"Эпоха {epoch:02d}/{epochs}: "
            f"train loss="
            f"{train_metrics['loss']:.5f}, "
            f"val loss="
            f"{val_metrics['loss']:.5f}, "
            f"val RMSE="
            f"{val_rmse_mw:.2f} МВт"
        )

        if (
            epochs_without_improvement
            >= patience
        ):
            print(
                "Early stopping"
            )
            break

    model.load_state_dict(
        best_state
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    return (
        model,
        history,
        best_val_rmse,
        elapsed,
    )


@torch.no_grad()
def predict_scaled(
    model,
    loader,
):
    model.eval()
    predictions = []
    targets = []

    for x_batch, y_batch in loader:
        batch_predictions = (
            model(
                x_batch.to(
                    DEVICE,
                    non_blocking=True,
                )
            )
            .cpu()
            .numpy()
        )

        predictions.append(
            batch_predictions
        )
        targets.append(
            y_batch.numpy()
        )

    return (
        np.concatenate(
            targets
        ),
        np.concatenate(
            predictions
        ),
    )


## 9. Подбор гиперпараметров

Сравниваются несколько конфигураций LSTM и GRU.
Все решения принимаются только по validation.

Критерий выбора — минимальный **validation RMSE в МВт**.
Test-выборка в подборе гиперпараметров не используется.


In [ ]:
TRIALS = [
    {
        "cell": "LSTM",
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.10,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
    },
    {
        "cell": "LSTM",
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "learning_rate": 5e-4,
        "weight_decay": 1e-4,
    },
    {
        "cell": "GRU",
        "hidden_size": 64,
        "num_layers": 1,
        "dropout": 0.15,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
    },
]

preview_model = make_model(TRIALS[0])
print(preview_model)
print(
    "Параметров:",
    sum(parameter.numel() for parameter in preview_model.parameters()),
)

In [ ]:
search_results = []

for trial_id, config in enumerate(
    TRIALS,
    start=1,
):
    print(
        f"\nЭксперимент "
        f"{trial_id}/{len(TRIALS)}: "
        f"{config}"
    )

    (
        model,
        trial_history,
        best_val_rmse,
        elapsed,
    ) = train_model(
        config=config,
        train_dl=search_loader,
        val_dl=val_loader,
        epochs=SEARCH_EPOCHS,
        patience=2,
        seed=SEED + trial_id,
    )

    best_epoch = int(
        np.argmin(
            trial_history[
                "val_rmse_mw"
            ]
        )
        + 1
    )

    search_results.append(
        {
            **config,
            "best_epoch": (
                best_epoch
            ),
            "best_val_rmse_mw": (
                best_val_rmse
            ),
            "seconds": elapsed,
            "parameters": sum(
                parameter.numel()
                for parameter
                in model.parameters()
            ),
        }
    )

    del model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

search_df = (
    pd.DataFrame(
        search_results
    )
    .sort_values(
        "best_val_rmse_mw"
    )
)

display(
    search_df.style.format(
        {
            "learning_rate": (
                "{:.1e}"
            ),
            "weight_decay": (
                "{:.1e}"
            ),
            "best_val_rmse_mw": (
                "{:.2f}"
            ),
            "seconds": "{:.1f}",
        }
    )
)

best_row = search_df.iloc[0]

best_config = {
    "cell": str(
        best_row["cell"]
    ),
    "hidden_size": int(
        best_row[
            "hidden_size"
        ]
    ),
    "num_layers": int(
        best_row[
            "num_layers"
        ]
    ),
    "dropout": float(
        best_row["dropout"]
    ),
    "learning_rate": float(
        best_row[
            "learning_rate"
        ]
    ),
    "weight_decay": float(
        best_row[
            "weight_decay"
        ]
    ),
}

print(
    "Выбрана конфигурация:",
    best_config,
)


## 10. Финальное обучение

Лучшая конфигурация заново обучается на всей обучающей выборке.

In [ ]:
CHECKPOINT_PATH = (
    ARTIFACT_DIR
    / "best_energy_rnn.pt"
)

(
    best_model,
    history,
    best_val_rmse,
    training_seconds,
) = train_model(
    config=best_config,
    train_dl=train_loader,
    val_dl=val_loader,
    epochs=MAX_EPOCHS,
    patience=4,
    checkpoint_path=(
        CHECKPOINT_PATH
    ),
    seed=SEED,
)

best_epoch = int(
    np.argmin(
        history[
            "val_rmse_mw"
        ]
    )
    + 1
)

parameter_count = sum(
    parameter.numel()
    for parameter
    in best_model.parameters()
)

run_config = {
    "run_id": RUN_ID,
    "seed": SEED,
    "target": TARGET,
    "features": FEATURES,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": (
        VAL_RATIO
    ),
    "best_config": best_config,
    "best_epoch": best_epoch,
    "best_val_rmse_mw": (
        best_val_rmse
    ),
    "parameter_count": (
        parameter_count
    ),
    "training_seconds": (
        training_seconds
    ),
}

(
    ARTIFACT_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

history_df = pd.DataFrame(
    history
)
history_df.to_csv(
    ARTIFACT_DIR
    / "history.csv",
    index=False,
)

print(
    "Лучшая эпоха:",
    best_epoch,
)
print(
    "Validation RMSE, МВт:",
    round(
        best_val_rmse,
        2,
    ),
)
print(
    "Время обучения, с:",
    round(
        training_seconds,
        1,
    ),
)
print(
    "Параметров модели:",
    parameter_count,
)


In [ ]:
epochs = np.arange(
    1,
    len(
        history[
            "train_loss"
        ]
    )
    + 1,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4),
)

axes[0].plot(
    epochs,
    history["train_loss"],
    marker="o",
    label="train",
)
axes[0].plot(
    epochs,
    history["val_loss"],
    marker="o",
    label="validation",
)
axes[0].axvline(
    best_epoch,
    linestyle="--",
    alpha=0.6,
)
axes[0].set_title(
    "Huber loss"
)
axes[0].set_xlabel(
    "Эпоха"
)
axes[0].legend()

axes[1].plot(
    epochs,
    history[
        "train_rmse_mw"
    ],
    marker="o",
    label="train",
)
axes[1].plot(
    epochs,
    history[
        "val_rmse_mw"
    ],
    marker="o",
    label="validation",
)
axes[1].axvline(
    best_epoch,
    linestyle="--",
    alpha=0.6,
)
axes[1].set_title(
    "RMSE, МВт"
)
axes[1].set_xlabel(
    "Эпоха"
)
axes[1].legend()

plt.tight_layout()
plt.savefig(
    ARTIFACT_DIR
    / "training_history.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 11. Оценка train, validation и test

Для лучшей RNN рассчитываются MAE, RMSE и R² отдельно на train,
validation и test после обратного масштабирования в мегаватты.

Test используется только после завершения выбора конфигурации.
Дополнительно на test модель сравнивается с двумя baseline:

- persistence: следующее значение равно последнему наблюдению;
- weekly seasonal: следующее значение равно значению неделю назад.

Прогноз является **одношаговым**: для каждого следующего часа
используется уже наблюдаемая прошлая история, а не рекурсивная
генерация всего test-периода.


In [ ]:
def inverse_target(values):
    values = np.asarray(
        values
    ).reshape(-1, 1)

    return (
        target_scaler
        .inverse_transform(
            values
        )
        .ravel()
    )


def smape(
    y_true,
    y_pred,
    epsilon=1e-8,
):
    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
        + epsilon
    )

    return 100 * np.mean(
        2
        * np.abs(
            y_pred - y_true
        )
        / denominator
    )


def safe_r2(
    y_true,
    y_pred,
):
    if np.allclose(
        y_true,
        y_true[0],
    ):
        return np.nan

    return r2_score(
        y_true,
        y_pred,
    )


def regression_metrics(
    y_true,
    y_pred,
):
    return {
        "MAE, MW": (
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "RMSE, MW": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        ),
        "R²": safe_r2(
            y_true,
            y_pred,
        ),
        "sMAPE, %": smape(
            y_true,
            y_pred,
        ),
    }


def evaluate_split(
    model,
    loader,
    dataset,
    split_name,
):
    (
        y_scaled,
        prediction_scaled,
    ) = predict_scaled(
        model,
        loader,
    )

    y_true = inverse_target(
        y_scaled
    )
    y_pred = inverse_target(
        prediction_scaled
    )

    timestamps = (
        series_df.index[
            dataset.target_indices
        ]
    )

    metrics = {
        "split": split_name,
        **regression_metrics(
            y_true,
            y_pred,
        ),
    }

    predictions = pd.DataFrame(
        {
            "run_id": RUN_ID,
            "split": split_name,
            "timestamp": (
                timestamps
            ),
            "y_true": y_true,
            "y_pred": y_pred,
            "residual": (
                y_true - y_pred
            ),
            "absolute_error": (
                np.abs(
                    y_true - y_pred
                )
            ),
        }
    )

    return metrics, predictions


split_evaluations = [
    (
        "train",
        train_eval_loader,
        train_dataset,
    ),
    (
        "validation",
        val_loader,
        val_dataset,
    ),
    (
        "test",
        test_loader,
        test_dataset,
    ),
]

metric_rows = []
prediction_tables = []

for (
    split_name,
    loader,
    dataset,
) in split_evaluations:
    (
        metrics,
        predictions,
    ) = evaluate_split(
        best_model,
        loader,
        dataset,
        split_name,
    )

    metric_rows.append(
        metrics
    )
    prediction_tables.append(
        predictions
    )

metrics_df = (
    pd.DataFrame(
        metric_rows
    )
    .set_index("split")
)

predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True,
)

metrics_df.to_csv(
    ARTIFACT_DIR
    / "metrics_by_split.csv"
)
predictions_df.to_csv(
    ARTIFACT_DIR
    / "predictions.csv",
    index=False,
)

display(
    metrics_df.style.format(
        "{:.4f}"
    )
)


In [ ]:
test_indices = (
    test_dataset
    .target_indices
)
test_time = (
    series_df.index[
        test_indices
    ]
)

test_predictions = (
    predictions_df[
        predictions_df[
            "split"
        ]
        == "test"
    ]
    .reset_index(
        drop=True
    )
)

y_test_true = (
    test_predictions[
        "y_true"
    ]
    .to_numpy()
)
y_test_pred = (
    test_predictions[
        "y_pred"
    ]
    .to_numpy()
)

y_persistence = (
    series_df[TARGET]
    .iloc[
        test_indices
        - HORIZON
    ]
    .to_numpy()
)

y_weekly = (
    series_df[TARGET]
    .iloc[
        test_indices
        - 168
    ]
    .to_numpy()
)

rnn_name = (
    f"{best_config['cell']} RNN"
)

test_results = pd.DataFrame(
    [
        {
            "model": (
                "Persistence baseline"
            ),
            **regression_metrics(
                y_test_true,
                y_persistence,
            ),
        },
        {
            "model": (
                "Weekly seasonal "
                "baseline"
            ),
            **regression_metrics(
                y_test_true,
                y_weekly,
            ),
        },
        {
            "model": rnn_name,
            **regression_metrics(
                y_test_true,
                y_test_pred,
            ),
        },
    ]
).set_index("model")

display(
    test_results.style.format(
        "{:.4f}"
    )
)

best_baseline_mae = (
    test_results.loc[
        [
            "Persistence baseline",
            "Weekly seasonal "
            "baseline",
        ],
        "MAE, MW",
    ]
    .min()
)

rnn_mae = test_results.loc[
    rnn_name,
    "MAE, MW",
]

improvement = (
    100
    * (
        best_baseline_mae
        - rnn_mae
    )
    / best_baseline_mae
)

print(
    "Изменение MAE относительно "
    "лучшего baseline:",
    f"{improvement:+.2f}%",
)

In [ ]:
plot_hours = 24 * 14

fig, ax = plt.subplots(
    figsize=(16, 5)
)

ax.plot(
    test_time[:plot_hours],
    y_test_true[:plot_hours],
    label="Факт",
    linewidth=1.8,
)

ax.plot(
    test_time[:plot_hours],
    y_test_pred[:plot_hours],
    label="RNN",
    linewidth=1.2,
)

ax.plot(
    test_time[:plot_hours],
    y_weekly[:plot_hours],
    label="Недельный baseline",
    linewidth=1.0,
    alpha=0.7,
)

ax.set_title(
    "Прогноз на первых "
    "двух неделях test"
)
ax.set_ylabel(
    "Энергопотребление, МВт"
)
ax.legend()

plt.tight_layout()
plt.savefig(
    ARTIFACT_DIR
    / "forecast_vs_actual.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 12. Анализ ошибок

Остаток определяется как факт минус прогноз. Для качественной модели остатки должны быть сосредоточены около нуля и не иметь выраженного систематического рисунка. В таблице отдельно показываются десять наиболее сложных часов.

In [ ]:
residuals = (
    y_test_true
    - y_test_pred
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4),
)

sns.histplot(
    residuals,
    bins=60,
    kde=True,
    ax=axes[0],
)
axes[0].axvline(
    0,
    linestyle="--",
)
axes[0].set_title(
    "Распределение остатков"
)
axes[0].set_xlabel(
    "Факт − прогноз, МВт"
)

axes[1].scatter(
    y_test_pred,
    residuals,
    s=8,
    alpha=0.25,
)
axes[1].axhline(
    0,
    linestyle="--",
)
axes[1].set_title(
    "Остатки и прогноз"
)
axes[1].set_xlabel(
    "Прогноз, МВт"
)
axes[1].set_ylabel(
    "Остаток, МВт"
)

plt.tight_layout()
plt.savefig(
    ARTIFACT_DIR
    / "residual_analysis.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

error_df = pd.DataFrame(
    {
        "timestamp": test_time,
        "actual_MW": y_test_true,
        "prediction_MW": (
            y_test_pred
        ),
        "weekly_baseline_MW": (
            y_weekly
        ),
        "residual_MW": (
            residuals
        ),
        "absolute_error_MW": (
            np.abs(
                residuals
            )
        ),
    }
).set_index(
    "timestamp"
)

hardest_hours = (
    error_df.nlargest(
        10,
        "absolute_error_MW",
    )
    .round(2)
)

display(
    hardest_hours
)

hardest_hours.to_csv(
    ARTIFACT_DIR
    / "largest_test_errors.csv"
)


In [ ]:
def predict_next_hour(
    recent_series,
):
    if len(recent_series) < LOOKBACK:
        raise ValueError(
            "Нужно не менее "
            f"{LOOKBACK} наблюдений"
        )

    prepared = (
        add_calendar_features(
            recent_series.iloc[
                -LOOKBACK:
            ]
        )
    )

    x_value = (
        feature_scaler
        .transform(
            prepared[FEATURES]
        )
        .astype("float32")
    )

    x_tensor = (
        torch.from_numpy(
            x_value
        )
        .unsqueeze(0)
        .to(DEVICE)
    )

    best_model.eval()

    with torch.no_grad():
        prediction_scaled = (
            best_model(
                x_tensor
            )
            .cpu()
            .numpy()
        )

    return float(
        inverse_target(
            prediction_scaled
        )[0]
    )


next_hour_prediction = (
    predict_next_hour(
        series_df.iloc[
            -LOOKBACK:
        ]
    )
)

print(
    "Прогноз после последней "
    "временной метки:",
    f"{next_hour_prediction:.1f} МВт",
)


In [ ]:
history_hours = 72

recent_history = (
    series_df[TARGET]
    .iloc[
        -history_hours:
    ]
)

last_timestamp = (
    recent_history.index[-1]
)
forecast_timestamp = (
    last_timestamp
    + pd.Timedelta(
        hours=HORIZON
    )
)
last_actual = (
    recent_history.iloc[-1]
)

plt.figure(
    figsize=(15, 5)
)

plt.plot(
    recent_history.index,
    recent_history.values,
    linewidth=1.8,
    marker="o",
    markersize=3,
    label=(
        "Фактическое "
        "энергопотребление"
    ),
)

plt.plot(
    [
        last_timestamp,
        forecast_timestamp,
    ],
    [
        last_actual,
        next_hour_prediction,
    ],
    linestyle="--",
    linewidth=2,
)

plt.scatter(
    forecast_timestamp,
    next_hour_prediction,
    s=120,
    zorder=5,
    label=(
        "Прогноз следующего часа"
    ),
)

plt.annotate(
    (
        f"{next_hour_prediction:.1f} "
        "МВт"
    ),
    xy=(
        forecast_timestamp,
        next_hour_prediction,
    ),
    xytext=(-110, 25),
    textcoords="offset points",
    arrowprops={
        "arrowstyle": "->",
    },
    fontsize=11,
    fontweight="bold",
)

plt.axvline(
    last_timestamp,
    linestyle=":",
    label=(
        "Граница известных данных"
    ),
)

plt.title(
    "Прогноз энергопотребления "
    "на следующий час"
)
plt.xlabel(
    "Дата и время"
)
plt.ylabel(
    "Энергопотребление, МВт"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    ARTIFACT_DIR
    / "next_hour_forecast.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 13. Итог и соответствие требованиям

В работе:

- выбран один домен — временной ряд энергопотребления;
- реализованы и сравниваются LSTM и GRU;
- используется хронологическое train/validation/test разбиение;
- scaler обучается только на train;
- окна не пересекают границы split;
- гиперпараметры выбираются только по validation;
- checkpoint выбирается по минимальному validation RMSE;
- сохраняются лучшие и текущие checkpoint;
- фиксируются random seed и параметры среды;
- MAE, RMSE и R² рассчитываются отдельно на train,
  validation и test;
- сохраняются scaler, конфигурация, история обучения,
  метрики и таблица предсказаний;
- построены графики обучения, факт/прогноз и остатки;
- показаны наиболее сложные прогнозы;
- выполнено сравнение с baseline;
- показан прогноз следующего часа.

Текстовая ветка не выполняется, поскольку выбран временной
домен. Сравнение с Transformer не включено, так как этот блок
является опциональным.


## 14. Источники

1. Hourly Energy Consumption, Kaggle, Version 3,
   лицензия CC0: Public Domain.
2. PJM Interconnection — исходный источник данных
   энергопотребления.
3. PyTorch — LSTM, GRU и средства обучения моделей.
4. scikit-learn — StandardScaler и метрики регрессии.
5. Материалы дисциплины
   «Прикладной искусственный интеллект».
